이 노트북은 GitHub 제출용 사본입니다. 원본 사진·첨부와 코드 출력은 포함하지 않습니다. 로컬 비교 그림은 PILL_INTERNAL_EDA_DIR 환경변수로 EDA 결과 폴더를 연결하세요.

In [ ]:
import sys
from pathlib import Path

project_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src" / "config.py").is_file()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [ ]:
from notebooks.eda_report_view import load_table, report_root, show_figure

# 내부 데이터 EDA

분석일 2026-11-04 · seed42

원본 이미지·JSON은 수정하지 않았습니다. 이 보고서는 현재 폴더의 관측 결과입니다.

| 항목 | 개수 |
| --- | --- |
| train_images | 232 |
| test_images | 842 |
| json_files | 763 |
| annotations | 763 |
| classes | 56 |
| train_groups | 114 |

## 1. 사진·라벨 구조

JSON 하나가 사진 하나를 뜻하지 않습니다. 알약별 JSON을 파일명으로 모아 사진 단위로 집계했습니다.

| 검사 | 개수 |
| --- | --- |
| image_decode_errors | 0 |
| json_errors | 0 |
| invalid_bbox | 1 |
| area_mismatch | 6 |
| image_size_mismatch | 0 |
| unmatched_image_join | 0 |
| unmatched_category_join | 0 |
| class_metadata_mismatch | 0 |
| train_images_without_any_json | 0 |
| json_image_without_photo | 0 |
| duplicate_image_hash_groups | 0 |
| train_test_exact_duplicate_groups | 0 |
| train_test_shared_name_groups | 0 |
| same_position_pairs | 3 |
| filename_missing_candidates | 8 |

### 먼저 확인할 결론

- 클래스별 bbox 수는 3~153개로 최대/최소 51배입니다.
- 촬영 조합이 하나뿐인 클래스는 17개입니다.
- 범위 밖 bbox·면적 불일치·서로 다른 클래스의 동일 위치 라벨을 먼저 검수하세요.
- null이 있는 행을 일괄 삭제하면 한 클래스 전체가 없어집니다.
- 단순 그룹 분할도 train에서 클래스가 빠질 수 있어 클래스 보존 예시를 함께 만들었습니다.

로컬 비교 그림: overview (다음 셀에서 표시)

In [ ]:
show_figure("internal", "overview.png")

로컬 비교 그림: classes (다음 셀에서 표시)

In [ ]:
show_figure("internal", "classes.png")

### 전체 클래스 분포

| class_id | pill_name | boxes | photos | groups | median_ratio | median_relative_area |
| --- | --- | --- | --- | --- | --- | --- |
| 3351 | 일양하이트린정 2mg | 153 | 153 | 89 | 0.9946236559139785 | 0.02769195056352459 |
| 3483 | 기넥신에프정(은행엽엑스)(수출용) | 45 | 45 | 15 | 1.2938388625592416 | 0.04483542520491803 |
| 35206 | 아토젯정 10/40mg | 37 | 37 | 16 | 0.6691729323308271 | 0.080859375 |
| 16262 | 크레스토정 20mg | 23 | 23 | 11 | 1.0254237288135593 | 0.04687740138319672 |
| 21325 | 아토르바정 10mg | 22 | 22 | 10 | 0.7846929705916451 | 0.0313780737704918 |
| 16232 | 리피토정 20mg | 21 | 21 | 11 | 1.0217391304347827 | 0.029352907274590164 |
| 3832 | 뉴로메드정(옥시라세탐) | 20 | 20 | 10 | 0.6596244131455399 | 0.10031658235143442 |
| 20238 | 플라빅스정 75mg | 20 | 20 | 11 | 1.0178187036594117 | 0.04106205174180328 |
| 36637 | 로수젯정10/5밀리그램 | 19 | 19 | 10 | 0.804 | 0.039291431864754096 |
| 16548 | 가바토파정 100mg | 18 | 18 | 6 | 1.0240917487684729 | 0.04334056416495902 |
| 29667 | 리바로정 4mg | 18 | 18 | 9 | 1.0178375685102141 | 0.041150902920081965 |
| 38162 | 로수바미브정 10/20mg | 18 | 18 | 10 | 0.8135461789400502 | 0.048528352330942624 |
| 19232 | 콜리네이트연질캡슐 400mg | 17 | 17 | 9 | 0.9967213114754099 | 0.07010357966188524 |
| 20014 | 마도파정 | 16 | 16 | 9 | 1.0310749299719888 | 0.09455726498463116 |
| 22074 | 리피로우정 20mg | 16 | 16 | 9 | 0.9815074054371868 | 0.02858526511270492 |
| 41768 | 카발린캡슐 25mg | 16 | 16 | 9 | 0.9221503831417625 | 0.06529360911885246 |
| 33880 | 글리틴정(콜린알포세레이트) | 16 | 16 | 9 | 1.2476681105176837 | 0.1058345607069672 |
| 13900 | 에빅사정(메만틴염산염)(비매품) | 15 | 15 | 7 | 0.701195219123506 | 0.03676997950819672 |
| 18147 | 리리카캡슐 150mg | 15 | 15 | 9 | 0.6866028708133971 | 0.08989898181352458 |
| 1900 | 보령부스파정 5mg | 15 | 15 | 5 | 1.0 | 0.02212554431352459 |
| 32310 | 글리아타민연질캡슐 | 14 | 14 | 9 | 0.9927231909581777 | 0.07588330878586065 |
| 18357 | 종근당글리아티린연질캡슐(콜린알포세레이트)  | 13 | 13 | 9 | 0.7804154302670623 | 0.07384733606557377 |
| 31863 | 아질렉트정(라사길린메실산염) | 12 | 12 | 9 | 0.99515712945591 | 0.034381803919057374 |
| 20877 | 엑스포지정 5/160mg | 12 | 12 | 4 | 1.3948972232239452 | 0.06199731045081967 |
| 25469 | 아모잘탄정 5/100mg | 12 | 12 | 4 | 1.4285853585358537 | 0.0715151927510246 |
| 27733 | 트윈스타정 40/5mg | 12 | 12 | 4 | 1.1374804936870477 | 0.06957647605020492 |
| 25367 | 자누메트정 50/850mg | 12 | 12 | 4 | 1.0991213700670142 | 0.14908627369364755 |
| 28763 | 트라젠타정(리나글립틴) | 9 | 9 | 3 | 1.045 | 0.034618340163934426 |
| 2483 | 뮤테란캡슐 100mg | 9 | 9 | 3 | 0.5063025210084033 | 0.09182569159836065 |
| 30308 | 트라젠타듀오정 2.5/850mg | 9 | 9 | 3 | 1.1424657534246576 | 0.1208952356557377 |
| 22347 | 자누비아정 50mg | 9 | 9 | 3 | 1.0406091370558375 | 0.03232662013319672 |
| 16688 | 오마코연질캡슐(오메가-3-산에틸에스테르90) | 8 | 8 | 7 | 0.7779403724310938 | 0.19170962474385245 |
| 19607 | 스토가정 10mg | 6 | 6 | 2 | 1.028726010616578 | 0.019665327228483607 |
| 3544 | 무코스타정(레바미피드)(비매품) | 6 | 6 | 2 | 1.0171103558954027 | 0.03379386526639344 |
| 19861 | 노바스크정 5mg | 6 | 6 | 2 | 1.287342650411339 | 0.03006371670081967 |
| 29345 | 비모보정 500/20mg | 6 | 6 | 2 | 0.6493497249422938 | 0.1015865138319672 |
| 25438 | 큐시드정 31.5mg/PTP | 6 | 6 | 2 | 0.9751144675031218 | 0.051744604892418035 |
| 12778 | 다보타민큐정 10mg/병 | 6 | 6 | 2 | 0.6002287214098239 | 0.13042952740778688 |
| 4543 | 에어탈정(아세클로페낙) | 5 | 5 | 2 | 1.0196078431372548 | 0.03396516393442623 |
| 3743 | 알드린정 | 3 | 3 | 1 | 0.9970059880239521 | 0.08325275358606557 |
| 19552 | 트루비타정 60mg/병 | 3 | 3 | 1 | 0.811529933481153 | 0.13212890625 |
| 16551 | 동아가바펜틴정 800mg | 3 | 3 | 1 | 0.6848739495798319 | 0.1167232325819672 |
| 12081 | 리렉스펜정 300mg/PTP | 3 | 3 | 1 | 0.5364806866952789 | 0.08766249359631148 |
| 12247 | 아빌리파이정 10mg | 3 | 3 | 1 | 0.9476439790575916 | 0.0266265368852459 |
| 13395 | 써스펜8시간이알서방정 650mg | 3 | 3 | 1 | 0.48541666666666666 | 0.09254930840163934 |
| 22362 | 맥시부펜이알정 300mg | 3 | 3 | 1 | 0.6097560975609756 | 0.08204725922131148 |
| 21771 | 라비에트정 20mg | 3 | 3 | 1 | 1.0317460317460319 | 0.02859006787909836 |
| 24850 | 놀텍정 10mg | 3 | 3 | 1 | 1.8047337278106508 | 0.040019050973360656 |
| 27993 | 졸로푸트정 100mg | 3 | 3 | 1 | 2.0306748466257667 | 0.0429047131147541 |
| 27926 | 울트라셋이알서방정 | 3 | 3 | 1 | 0.5348360655737705 | 0.09879290471311475 |
| 27777 | 카나브정 60mg | 3 | 3 | 1 | 0.8838383838383839 | 0.02773597592213115 |
| 29451 | 레일라정 | 3 | 3 | 1 | 1.4848484848484849 | 0.08548363857581967 |
| 31885 | 자누메트엑스알서방정 100/1000mg | 3 | 3 | 1 | 0.7383720930232558 | 0.16233350409836064 |
| 34597 | 제미메트서방정 50/1000mg | 3 | 3 | 1 | 1.1404358353510895 | 0.15161533043032788 |
| 33009 | 신바로정 | 3 | 3 | 1 | 1.8229665071770336 | 0.06123046875 |
| 33208 | 에스원엠프정 20mg | 3 | 3 | 1 | 1.0495049504950495 | 0.03427894467213115 |

### 클래스 불균형과 추가 정합성

| 항목 | 값 |
| --- | --- |
| min_bbox_per_class | 3.0 |
| max_bbox_per_class | 153.0 |
| max_min_ratio | 51.0 |
| largest_class_id | 3351.0 |
| largest_class_fraction | 0.20052424639580602 |
| classes_with_one_group | 17.0 |

소수 클래스는 촬영 조합 수도 함께 확인해야 합니다. 오버샘플링·증강은 train에만 적용하고 검증은 원래 분포에서 클래스별로 평가하세요.

| 검사 | 개수 |
| --- | --- |
| invalid_nonempty_dates | 0 |
| duplicate_annotation_id_rows | 0 |
| image_id_filename_conflicts | 0 |
| catalog_nonempty_value_conflicts | 0 |
| long_short_order_conflicts | 0 |

## 2. 결측 피처와 의미

| section | field | records | key_missing | null | empty_string | whitespace | empty_list | empty_object | text_missing_token | present |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| images | change_date | 763 | 0 | 18 | 0 | 0 | 0 | 0 | 0 | 745 |
| images | chart | 763 | 0 | 18 | 0 | 0 | 0 | 0 | 0 | 745 |
| images | color_class1 | 763 | 0 | 18 | 0 | 0 | 0 | 0 | 0 | 745 |
| images | color_class2 | 763 | 0 | 18 | 693 | 0 | 0 | 0 | 0 | 52 |
| images | di_company_mf | 763 | 0 | 0 | 637 | 0 | 0 | 0 | 0 | 126 |
| images | di_company_mf_en | 763 | 0 | 0 | 724 | 0 | 0 | 0 | 0 | 39 |
| images | dl_material | 763 | 0 | 0 | 9 | 0 | 0 | 0 | 0 | 754 |
| images | dl_name_en | 763 | 0 | 0 | 14 | 0 | 0 | 0 | 0 | 749 |
| images | drug_shape | 763 | 0 | 18 | 0 | 0 | 0 | 0 | 0 | 745 |
| images | form_code_name | 763 | 0 | 18 | 0 | 0 | 0 | 0 | 0 | 745 |
| images | img_key | 763 | 0 | 18 | 0 | 0 | 0 | 0 | 0 | 745 |
| images | img_regist_ts | 763 | 0 | 18 | 0 | 0 | 0 | 0 | 0 | 745 |
| images | line_back | 763 | 0 | 18 | 630 | 0 | 0 | 0 | 0 | 115 |
| images | line_front | 763 | 0 | 18 | 552 | 0 | 0 | 0 | 0 | 193 |
| images | mark_code_back | 763 | 0 | 18 | 745 | 0 | 0 | 0 | 0 | 0 |
| images | mark_code_back_anal | 763 | 0 | 18 | 745 | 0 | 0 | 0 | 0 | 0 |
| images | mark_code_back_img | 763 | 0 | 18 | 745 | 0 | 0 | 0 | 0 | 0 |
| images | mark_code_front | 763 | 0 | 18 | 745 | 0 | 0 | 0 | 0 | 0 |
| images | mark_code_front_anal | 763 | 0 | 18 | 642 | 0 | 0 | 0 | 0 | 103 |
| images | mark_code_front_img | 763 | 0 | 18 | 642 | 0 | 0 | 0 | 0 | 103 |
| images | print_back | 763 | 0 | 18 | 256 | 0 | 0 | 0 | 0 | 489 |
| images | print_front | 763 | 0 | 18 | 16 | 0 | 0 | 0 | 0 | 729 |
| annotations | segmentation | 763 | 0 | 0 | 0 | 0 | 763 | 0 | 0 | 0 |

null·빈 문자열·키 부재·빈 리스트를 구분했습니다. segmentation=[]는 bbox 전용 라벨에 마스크가 없는 상태이며, bbox 결측과 다릅니다.

| metadata_rows | rows_with_null | classes_lost_by_null_drop | rows_left_after_null_drop | rows_left_after_blank_and_null_drop |
| --- | --- | --- | --- | --- |
| 763 | 18 | [16548] | 745 | 0 |

**행 일괄 삭제를 피하세요.** 설명이 비어도 사진·클래스·bbox는 존재합니다. 원인은 제품 정보 수집·연계 누락 또는 선택 필드 미기재일 수 있으나 결측 패턴만으로 생성 원인이나 MCAR/MAR/MNAR를 확정하지 않습니다.

로컬 비교 그림: missing (다음 셀에서 표시)

In [ ]:
show_figure("internal", "missing.png")

### 결측 처리 판단

| 대상 | 의미·처리 |
|---|---|
| print_front/back | 빈 값 ≠ 무각인. 실제 관측면·OCR 정답 별도 검수 |
| color_class2·마크 | 미기재 ≠ 실제 부재. 주색·다른 제품 최빈값 복사 금지 |
| thick/leng_long/leng_short=0 | 물리량에서 NA+invalid. 카메라0·ignore0 유지 |
| 전체 미기재 열 | 현재 모델 입력에서 제외 가능. 원본 스키마·행 보존 |
| 색·모양 보조 정답 결측 | 해당 속성 loss만 제외, 검수된 검출 정답은 유지 |
| 클래스별 전체 결측 | 클래스 내부 대체 불가. 검증된 동일 제품 출처로만 보충 |

제품 설명은 클래스에 따라 일정할 수 있습니다. 정답 제품 카탈로그나 결측 마스크를 사진 입력에 붙이면 클래스 누수가 생깁니다. 이미지 전용 기준 모델 → 보조 속성 예측 → 사진에서 읽은 OCR과 전체 후보 카탈로그 비교 순서로 실험할 수 있습니다. drug_dir가 앞면이어도 카탈로그 앞 각인이 보인다고 가정하지 않습니다.

## 3. bbox·면적·물리 치수 이상치

| annotation_id | class_id | pill_name | bbox | area | bbox_area | area_delta | bbox_valid |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 3019 | 12778 | 다보타민큐정 10mg/병 | [88, 48, 317, 508] | 161096 | 161036 | 60 | True |
| 903 | 18357 | 종근당글리아티린연질캡슐(콜린알포세레이트)  | [6567, 625, 311, 315] | 97965 | 97965 | 0 | False |
| 5178 | 19232 | 콜리네이트연질캡슐 400mg | [523, 201, 232, 241] | 79112 | 55912 | 23200 | True |
| 2633 | 19232 | 콜리네이트연질캡슐 400mg | [141, 801, 322, 266] | 88312 | 85652 | 2660 | True |
| 3000 | 3351 | 일양하이트린정 2mg | [427, 839, 189, 182] | 36288 | 34398 | 1890 | True |
| 5346 | 33880 | 글리틴정(콜린알포세레이트) | [558, 195, 309, 420] | 129480 | 129780 | -300 | True |
| 5351 | 38162 | 로수바미브정 10/20mg | [129, 269, 251, 259] | 62009 | 65009 | -3000 | True |

범위 밖 bbox는 원본 좌표 복구 또는 검수 전 격리가 필요합니다. 화면에 강제로 자르거나 평균 좌표를 넣지 않습니다. area 불일치는 사진 기준 bbox 검수·복구 후 최종 w×h로 재계산합니다. 수치가 유효해도 알약 몸통을 자르거나 다른 알약을 가리킬 수 있습니다.

| class_id | field | value | lower | upper | reason |
| --- | --- | --- | --- | --- | --- |
| 16548 | thick | 0.0 | 0.3174999999999999 | 9.7775 | 물리 치수의 0·결측·비유한 값 |
| 16688 | thick | 9.8 | 0.3174999999999999 | 9.7775 | 클래스별 대표값 IQR 후보 (오류 확정 아님) |
| 16548 | leng_long | 0.0 | -5.562499999999998 | 31.577499999999997 | 물리 치수의 0·결측·비유한 값 |
| 16548 | leng_short | 0.0 | 2.2500000000000018 | 13.049999999999999 | 물리 치수의 0·결측·비유한 값 |

물리 치수 IQR은 양수인 클래스별 대표값에 적용했습니다. 반복 촬영 행을 독립 약품으로 세지 않았습니다. 단위·측정 근거 없이 큰 알약을 삭제하거나 평균으로 바꾸지 않습니다. bbox 비율은 회전·각도 영향도 받습니다.

로컬 비교 그림: bbox (다음 셀에서 표시)

In [ ]:
show_figure("internal", "bbox.png")

## 4. 중복·누락·클래스 연결 후보

| file_name | first_id | second_id | first_class | second_class | iou | same_class |
| --- | --- | --- | --- | --- | --- | --- |
| K-001900-016548-019607-033009_0_2_0_2_70_000_200.png | 4683 | 4691 | 16548 | 33009 | 1.0 | False |
| K-003351-020238-031863_0_2_0_2_70_000_200.png | 3444 | 3441 | 3351 | 20238 | 1.0 | False |
| K-003351-029667-031863_0_2_0_2_70_000_200.png | 791 | 788 | 3351 | 31863 | 1.0 | False |

| file_name | expected_class | reason | visually_confirmed |
| --- | --- | --- | --- |
| K-003351-013900-021325_0_2_0_2_70_000_200.png | 3351 | 파일명에 있으나 현재 JSON 클래스 연결 없음 | False |
| K-003351-013900-036637_0_2_0_2_70_000_200.png | 3351 | 파일명에 있으나 현재 JSON 클래스 연결 없음 | False |
| K-003351-020014-022074_0_2_0_2_90_000_200.png | 20014 | 파일명에 있으나 현재 JSON 클래스 연결 없음 | False |
| K-003351-021325-032310_0_2_0_2_90_000_200.png | 32310 | 파일명에 있으나 현재 JSON 클래스 연결 없음 | False |
| K-003351-032310-038162_0_2_0_2_70_000_200.png | 3351 | 파일명에 있으나 현재 JSON 클래스 연결 없음 | False |
| K-003351-033880-038162_0_2_0_2_75_000_200.png | 33880 | 파일명에 있으나 현재 JSON 클래스 연결 없음 | False |
| K-003351-035206-041768_0_2_0_2_70_000_200.png | 3351 | 파일명에 있으나 현재 JSON 클래스 연결 없음 | False |
| K-003544-004543-012247-016548_0_2_0_2_90_000_200.png | 4543 | 파일명에 있으나 현재 JSON 클래스 연결 없음 | False |

동일 위치의 다른 클래스는 충돌 후보입니다. 이름·각인·동일 클래스 참조로 검수합니다. 파일명 기대 클래스가 JSON에 없으면 **누락 후보**이며 파일명만으로 복구하지 않습니다. 박스 없이 보이는 알약은 검출 학습에서 배경으로 학습될 수 있으므로 원본 복구 전 사진 제외 또는 지원되는 무시 영역을 검토합니다. 자동 검사로 의미상 오라벨·몸통 잘림을 전수 확정한 것은 아닙니다.

## 5. 같은 유형끼리 가로 비교

한 줄에 최대 세 장을 배치했습니다. 클래스 ID·약 이름은 원본 라벨이며 오류 후보에서 실제 약품의 보증이 아닙니다. 참조 사진도 정상 라벨 인증으로 해석하지 않습니다.

### 화면 밖 bbox · 같은 클래스의 다른 촬영과 비교

현재 원본 라벨 표시 · 클래스·좌표 오류와 설명 결측을 구분

로컬 비교 그림: gallery_01 (다음 셀에서 표시)

In [ ]:
show_figure("internal", "gallery_01.jpg")

### 면적 불일치 · 클래스 12778

현재 원본 라벨 표시 · 클래스·좌표 오류와 설명 결측을 구분

로컬 비교 그림: gallery_02 (다음 셀에서 표시)

In [ ]:
show_figure("internal", "gallery_02.jpg")

### 면적 불일치 · 클래스 19232

현재 원본 라벨 표시 · 클래스·좌표 오류와 설명 결측을 구분

로컬 비교 그림: gallery_03 (다음 셀에서 표시)

In [ ]:
show_figure("internal", "gallery_03.jpg")

### 면적 불일치 · 클래스 19232

현재 원본 라벨 표시 · 클래스·좌표 오류와 설명 결측을 구분

로컬 비교 그림: gallery_04 (다음 셀에서 표시)

In [ ]:
show_figure("internal", "gallery_04.jpg")

### 면적 불일치 · 클래스 3351

현재 원본 라벨 표시 · 클래스·좌표 오류와 설명 결측을 구분

로컬 비교 그림: gallery_05 (다음 셀에서 표시)

In [ ]:
show_figure("internal", "gallery_05.jpg")

### 면적 불일치 · 클래스 33880

현재 원본 라벨 표시 · 클래스·좌표 오류와 설명 결측을 구분

로컬 비교 그림: gallery_06 (다음 셀에서 표시)

In [ ]:
show_figure("internal", "gallery_06.jpg")

### 면적 불일치 · 클래스 38162

현재 원본 라벨 표시 · 클래스·좌표 오류와 설명 결측을 구분

로컬 비교 그림: gallery_07 (다음 셀에서 표시)

In [ ]:
show_figure("internal", "gallery_07.jpg")

### 설명 결측 · 사진과 bbox가 없는 뜻이 아님

현재 원본 라벨 표시 · 클래스·좌표 오류와 설명 결측을 구분

로컬 비교 그림: gallery_08 (다음 셀에서 표시)

In [ ]:
show_figure("internal", "gallery_08.jpg")

### 각인 빈 문자열 · 실제 사진의 문자는 별도 확인

현재 원본 라벨 표시 · 클래스·좌표 오류와 설명 결측을 구분

로컬 비교 그림: gallery_09 (다음 셀에서 표시)

In [ ]:
show_figure("internal", "gallery_09.jpg")

### 물리 치수 IQR 후보 · 같은 제품의 여러 촬영

현재 원본 라벨 표시 · 클래스·좌표 오류와 설명 결측을 구분

로컬 비교 그림: gallery_10 (다음 셀에서 표시)

In [ ]:
show_figure("internal", "gallery_10.jpg")

### 동일 위치·다른 클래스 · 중복 정답 충돌 후보

IoU=1.000 · 실제 클래스는 이미지와 원본 대조로 판정

로컬 비교 그림: gallery_11 (다음 셀에서 표시)

In [ ]:
show_figure("internal", "gallery_11.jpg")

### 동일 위치·다른 클래스 · 중복 정답 충돌 후보

IoU=1.000 · 실제 클래스는 이미지와 원본 대조로 판정

로컬 비교 그림: gallery_12 (다음 셀에서 표시)

In [ ]:
show_figure("internal", "gallery_12.jpg")

### 동일 위치·다른 클래스 · 중복 정답 충돌 후보

IoU=1.000 · 실제 클래스는 이미지와 원본 대조로 판정

로컬 비교 그림: gallery_13 (다음 셀에서 표시)

In [ ]:
show_figure("internal", "gallery_13.jpg")

### 라벨 연결 누락 후보 · 기대 클래스 3351

후보 장면의 기존 박스 표시 · 카드 이름은 원본 라벨

로컬 비교 그림: gallery_14 (다음 셀에서 표시)

In [ ]:
show_figure("internal", "gallery_14.jpg")

### 라벨 연결 누락 후보 · 기대 클래스 3351

후보 장면의 기존 박스 표시 · 카드 이름은 원본 라벨

로컬 비교 그림: gallery_15 (다음 셀에서 표시)

In [ ]:
show_figure("internal", "gallery_15.jpg")

### 라벨 연결 누락 후보 · 기대 클래스 4543

후보 장면의 기존 박스 표시 · 카드 이름은 원본 라벨

로컬 비교 그림: gallery_16 (다음 셀에서 표시)

In [ ]:
show_figure("internal", "gallery_16.jpg")

### 라벨 연결 누락 후보 · 기대 클래스 20014

후보 장면의 기존 박스 표시 · 카드 이름은 원본 라벨

로컬 비교 그림: gallery_17 (다음 셀에서 표시)

In [ ]:
show_figure("internal", "gallery_17.jpg")

### 라벨 연결 누락 후보 · 기대 클래스 32310

후보 장면의 기존 박스 표시 · 카드 이름은 원본 라벨

로컬 비교 그림: gallery_18 (다음 셀에서 표시)

In [ ]:
show_figure("internal", "gallery_18.jpg")

### 라벨 연결 누락 후보 · 기대 클래스 33880

후보 장면의 기존 박스 표시 · 카드 이름은 원본 라벨

로컬 비교 그림: gallery_19 (다음 셀에서 표시)

In [ ]:
show_figure("internal", "gallery_19.jpg")

## 6. 학습·검증 분할

| 지표 | 값 |
| --- | --- |
| seed | 42 |
| cross_group_exact_duplicates | 0 |
| random_split_overlapping_groups | 35 |
| random_split_train_photos | 186 |
| random_split_train_classes | 56 |
| random_split_valid_photos | 46 |
| random_split_valid_classes | 44 |
| group_split_overlapping_groups | 0 |
| group_split_train_photos | 189 |
| group_split_train_classes | 52 |
| group_split_valid_photos | 43 |
| group_split_valid_classes | 32 |
| coverage_group_split_overlapping_groups | 0 |
| coverage_group_split_train_photos | 185 |
| coverage_group_split_train_classes | 56 |
| coverage_group_split_valid_photos | 47 |
| coverage_group_split_valid_classes | 28 |
| coverage_search_candidates | 2000 |
| all_train_classes_retained | True |

무작위 사진 분할과 파일명 촬영 조합을 묶은 80:20 예시를 비교했습니다. 실제 학습은 하지 않았습니다. 최종 분할은 클래스별 대표성도 확인해야 합니다. coverage_group_split은 2,000개 후보 중 train 클래스 보존을 우선한 예시입니다. 검증에 없는 클래스는 이 분할로 성능을 평가할 수 없습니다. 정확 중복이 다른 그룹에 있으면 해시와 촬영 조합의 연결 성분을 함께 묶어야 합니다. 테스트는 정답이 없어 파일명 그룹·정확 바이트 중복만 비교했습니다. 파일명 그룹 겹침은 분포 중첩의 근거이며 테스트 정답 누수의 확정은 아닙니다. 같은 제품의 새 촬영 평가와 새로운 제품 일반화 평가는 클래스 분할 목적이 다릅니다.

## 7. 재현과 확인 범위

tables 폴더에 전체 행·출처·진단 표, figures 폴더에 그림, summary.json에 요약과 원본 manifest 해시를 저장했습니다. 파일 바이트 해시 중복 검사는 재인코딩된 시각적 유사 사진을 검출하지 못합니다. 밝기·gradient_energy는 장면 전체의 기술 지표이며 흐림·불량 판정이 아닙니다. 독립적인 약품 카탈로그 검증·OCR 모델·모델 학습은 수행하지 않았습니다.

현재 분석 폴더에서 재생성:

```powershell
uv run python analysis.py --data-dir ../../datasets/internal_raw --output-dir .
```

### 결과 표 다시 탐색하기 (선택)

위 내용과 이미지는 셀 실행 없이 볼 수 있습니다. 아래 셀은 이 노트북 폴더를 작업 디렉터리로 선택한 뒤 실행하세요.

In [ ]:
if (report_root("internal") / "tables" / "annotations.csv").is_file():
    bbox_df = load_table("internal", "annotations.csv")
    display(bbox_df.head().T)
else:
    print("PILL_INTERNAL_EDA_DIR을 로컬 EDA 결과 폴더로 설정하세요.")